# 9 Expert Angular-Spectrum Global Router Step-by-Step

这个 notebook 用来验证一个全程基于 Angular Spectrum Propagation 的 9-expert optical prompt router。

主光路是：input plane -> angular spectrum propagation z1 -> prompt plane -> complex-amplitude prompt router -> angular spectrum propagation z2 -> expert entrance plane -> optional identity expert propagation。

关键约束：expert_entrance 必须由 prop_prompt_to_expert(field_after_prompt) 得到，不使用 FFT convolution 公式直接生成 expert entrance。

和已有 notebook 的区别：

- nine_expert_prompt_step_by_step.ipynb 使用角谱法，但 prompt 是空间分区；每个分区只能调制照到该区域的光。
- nine_expert_global_fanout_prompt_step_by_step.ipynb 可以得到 9 个复制方向，但 expert entrance 来自 FFT convolution 公式。
- 本 notebook 输入仍然是中心 200 x 200，不扩展、不切 patch；router 的 amplitude 和 phase 都在 prompt 平面实现；expert entrance 由角谱传播得到。

为什么 f = 10 cm：默认 z1 = z2 = 20 cm。薄透镜成像关系 1/f = 1/z1 + 1/z2，所以 f = 10 cm，对应 1:1 成像。

为什么 grating frequency 用 z2：线性光栅在 prompt 平面引入横向波矢，小角度近似下 dx = z2 * wavelength * fx。因此 routing 位移由 z2 和 grating spatial frequency 决定。

为什么 amplitude 是 prompt-plane router：在主模式 complex_order_router 中，R_router(x,y) = normalize(sum_j a_j exp(i 2pi (fx_j x + fy_j y)))，a_j 直接进入 prompt transmission，不是在 expert entrance 后乘 gate。

为什么需要 calibration：角谱传播、采样、符号约定和有限孔径可能导致理论 shift 和实测 shift 不一致，所以用 onehot centroid sweep 选择 scale/sign。

In [ ]:
%matplotlib inline
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.patches import Rectangle

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd() / 'opticalmoe']
PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'src' / 'opticalmoe').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot find opticalmoe/src. Please run this notebook inside the repository.')
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from opticalmoe.optics.angular_spectrum import AngularSpectrumPropagator

torch.set_grad_enabled(False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR = PROJECT_ROOT / 'runs' / 'nine_expert_as_global_router_step_by_step'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11, 'figure.dpi': 130})
print('PROJECT_ROOT =', PROJECT_ROOT)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('device =', device)


## 1. Geometry and Physical Parameters


In [ ]:
WAVELENGTH_M = 532e-9
PIXEL_SIZE_M = 8e-6
INPUT_SIZE = 200
EXPERT_SIZE = 200
EXPERT_PITCH = 200
BANK_SIZE = 3 * EXPERT_SIZE
PADDING = 300
CANVAS_SIZE = BANK_SIZE + 2 * PADDING
CANVAS_SHAPE = (CANVAS_SIZE, CANVAS_SIZE)
CANVAS_CENTER = (CANVAS_SIZE // 2, CANVAS_SIZE // 2)
EXPERT_COORDS = [PADDING + EXPERT_SIZE // 2, PADDING + 3 * EXPERT_SIZE // 2, PADDING + 5 * EXPERT_SIZE // 2]
EXPERT_IDS = ['E00','E01','E02','E10','E11','E12','E20','E21','E22']

INPUT_TO_PROMPT_CM = 20.0
PROMPT_TO_EXPERT_CM = 20.0
INTER_LAYER_CM = 5.0
NUM_IDENTITY_EXPERT_LAYERS = 5
FOCAL_LENGTH_CM = 10.0

PROMPT_MODE = 'complex_order_router'  # complex_order_router / phase_only_angle_sum / region_amplitude_angle_sum
PROMPT_APERTURE_MODE = 'center_600'   # center_600 / full_canvas
INPUT_TYPE = 'mnist_or_synthetic_200'
AMPLITUDE_CASE = 'uniform'
GRATING_SCALE = 1.0
GRATING_SIGN_X = 1.0
GRATING_SIGN_Y = 1.0

# Full diagnostics can take time on CPU because each propagation uses 1200x1200 complex FFTs.
RUN_CALIBRATION = True
RUN_ONEHOT_MATRIX = True
RUN_AMPLITUDE_TESTS = True
RUN_COPY_SIMILARITY = True
RUN_DRIFT_ANALYSIS = True
SAVE_FIGURES = True

z1_m = INPUT_TO_PROMPT_CM * 1e-2
z2_m = PROMPT_TO_EXPERT_CM * 1e-2
f_m = FOCAL_LENGTH_CM * 1e-2
magnification = z2_m / z1_m
expected_image_size_px = INPUT_SIZE * magnification
pitch_m = EXPERT_PITCH * PIXEL_SIZE_M
f_pitch = pitch_m / (WAVELENGTH_M * z2_m)
period_pitch_px = 1.0 / (f_pitch * PIXEL_SIZE_M)
print('canvas:', CANVAS_SHAPE)
print('expert centers:', EXPERT_COORDS)
print('z1 input_to_prompt cm:', INPUT_TO_PROMPT_CM)
print('z2 prompt_to_expert cm:', PROMPT_TO_EXPERT_CM)
print('f focal_length cm:', FOCAL_LENGTH_CM)
print('magnification z2/z1:', magnification)
print('expected_image_size_px:', expected_image_size_px)
print('theoretical pitch frequency cycles/m:', f_pitch)
print('theoretical grating period px for 200 px pitch:', period_pitch_px)


In [ ]:
def cm_to_m(value): return float(value) * 1e-2

def centered_slice(center, size):
    half = size // 2
    return slice(center - half, center + half)

def make_grid_slices(size):
    apertures, centers = [], []
    for y in EXPERT_COORDS:
        for x in EXPERT_COORDS:
            apertures.append((centered_slice(y, size), centered_slice(x, size)))
            centers.append((y, x))
    return apertures, centers

EXPERT_APERTURES, EXPERT_CENTERS = make_grid_slices(EXPERT_SIZE)
PROMPT_APERTURES, PROMPT_CENTERS = make_grid_slices(EXPERT_SIZE)
INPUT_APERTURE = (centered_slice(CANVAS_CENTER[0], INPUT_SIZE), centered_slice(CANVAS_CENTER[1], INPUT_SIZE))

def make_mask(apertures, dtype=torch.float32):
    mask = torch.zeros(CANVAS_SHAPE, dtype=dtype, device=device)
    for ys, xs in apertures:
        mask[ys, xs] = 1
    return mask

EXPERT_UNION_MASK = make_mask(EXPERT_APERTURES)
CENTER_600_MASK = make_mask(PROMPT_APERTURES)
FULL_CANVAS_MASK = torch.ones(CANVAS_SHAPE, dtype=torch.float32, device=device)
yy_m = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[0]) * PIXEL_SIZE_M
xx_m = (torch.arange(CANVAS_SIZE, device=device).float() - CANVAS_CENTER[1]) * PIXEL_SIZE_M
Y_M, X_M = torch.meshgrid(yy_m, xx_m, indexing='ij')

def prompt_aperture_mask(mode=None):
    mode = PROMPT_APERTURE_MODE if mode is None else mode
    if mode == 'center_600': return CENTER_600_MASK
    if mode == 'full_canvas': return FULL_CANVAS_MASK
    raise ValueError(mode)

def save_json(path, payload):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f: json.dump(payload, f, indent=2)

def save_csv(path, rows, fieldnames=None):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        fieldnames = []
        for row in rows:
            for key in row:
                if key not in fieldnames: fieldnames.append(key)
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader(); writer.writerows(rows)

def plot_layout():
    fig, ax = plt.subplots(figsize=(7, 7)); ax.set_facecolor('black')
    ys, xs = INPUT_APERTURE
    ax.add_patch(Rectangle((xs.start, ys.start), INPUT_SIZE, INPUT_SIZE, fill=False, edgecolor='orange', linewidth=2, label='input'))
    if PROMPT_APERTURE_MODE == 'center_600': ax.add_patch(Rectangle((PADDING, PADDING), BANK_SIZE, BANK_SIZE, fill=False, edgecolor='cyan', linestyle='--', linewidth=1.5, label='prompt aperture'))
    else: ax.add_patch(Rectangle((0, 0), CANVAS_SIZE, CANVAS_SIZE, fill=False, edgecolor='cyan', linestyle='--', linewidth=1.5, label='prompt aperture'))
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor='lime', linewidth=1.1))
        ax.text(xs.start + 8, ys.start + 20, EXPERT_IDS[idx], color='lime', fontsize=9)
    ax.axhline(CANVAS_CENTER[0], color='white', alpha=0.25); ax.axvline(CANVAS_CENTER[1], color='white', alpha=0.25)
    ax.set_xlim(0, CANVAS_SIZE); ax.set_ylim(CANVAS_SIZE, 0)
    ax.set_title('1200x1200 canvas: input, prompt aperture, 9 expert apertures'); ax.legend(loc='upper right')
    if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'layout_overlay.png', bbox_inches='tight')
    plt.show()
plot_layout()


## 2. Input Construction

输入始终是中心 200 x 200，嵌入 1200 x 1200 canvas。不扩大输入，也不切成 9 个 patch。


In [ ]:
def synthetic_digit(size=200):
    img = torch.zeros(size, size, dtype=torch.float32)
    t, m = max(8, size // 13), max(12, size // 7)
    img[m:m+t, m:size-m] = 1.0
    for i in range(size // 2):
        y = m + i; x = size - m - t - int(i * 0.9)
        if 0 <= y < size and 0 <= x < size - t: img[y:y+t, x:x+t] = 1.0
    img[int(size*0.65):int(size*0.72), int(size*0.32):int(size*0.55)] = 0.55
    return img.clamp(0, 1)

def flat_top(size=200): return torch.ones(size, size, dtype=torch.float32)

def gaussian(size=200, sigma=None):
    sigma = size / 4 if sigma is None else sigma
    coords = torch.arange(size).float() - (size - 1) / 2
    yy, xx = torch.meshgrid(coords, coords, indexing='ij')
    return torch.exp(-(xx**2 + yy**2) / (2 * sigma**2)).float()

def load_mnist_or_synthetic(index=0, size=200):
    try:
        from torchvision.datasets import MNIST
        dataset = MNIST(root=str(PROJECT_ROOT / 'data'), train=False, download=False)
        img = dataset.data[index].float() / 255.0
        img = F.interpolate(img[None, None], size=(size, size), mode='bilinear', align_corners=False)[0, 0]
        print('Loaded MNIST sample, label =', int(dataset.targets[index])); return img
    except Exception as exc:
        print('MNIST unavailable, using synthetic digit. Reason:', repr(exc)); return synthetic_digit(size)

def make_input_image(input_type=INPUT_TYPE, size=INPUT_SIZE):
    if input_type == 'digit_like_200': return synthetic_digit(size)
    if input_type == 'mnist_or_synthetic_200': return load_mnist_or_synthetic(0, size)
    if input_type == 'flat_top_200': return flat_top(size)
    if input_type == 'gaussian_200': return gaussian(size)
    raise ValueError(input_type)

def embed_input(img):
    field = torch.zeros((1, CANVAS_SIZE, CANVAS_SIZE), dtype=torch.complex64, device=device)
    ys, xs = INPUT_APERTURE
    field[0, ys, xs] = img.to(device=device, dtype=torch.float32).to(torch.complex64)
    return field

input_img = make_input_image(INPUT_TYPE, INPUT_SIZE).to(device)
input_field = embed_input(input_img)
plt.figure(figsize=(4, 4)); plt.imshow(input_img.detach().cpu(), cmap='gray'); plt.title(f'input amplitude: {INPUT_TYPE}'); plt.axis('off')
if SAVE_FIGURES: plt.savefig(OUTPUT_DIR / 'input_plane_intensity.png', bbox_inches='tight')
plt.show()


## 3. Angular Spectrum Propagators

下面三个 propagator 是主光路唯一的传播算子。expert_entrance 不由 FFT convolution 直接生成。


In [ ]:
def make_propagator(distance_cm):
    return AngularSpectrumPropagator(wavelength_m=WAVELENGTH_M, pixel_size_m=PIXEL_SIZE_M, grid_size=CANVAS_SHAPE, distance_m=cm_to_m(distance_cm)).to(device)
prop_input_to_prompt = make_propagator(INPUT_TO_PROMPT_CM)
prop_prompt_to_expert = make_propagator(PROMPT_TO_EXPERT_CM)
prop_inter_layer = make_propagator(INTER_LAYER_CM)
print('prop_input_to_prompt:', type(prop_input_to_prompt).__name__)
print('prop_prompt_to_expert:', type(prop_prompt_to_expert).__name__)
print('prop_inter_layer:', type(prop_inter_layer).__name__)


## 4. Visualization and Diagnostics Helpers


In [ ]:
def intensity(field): return torch.abs(field).square() if torch.is_complex(field) else field.float()
def to_numpy_2d(tensor):
    if tensor.ndim == 3: tensor = tensor[0]
    return tensor.detach().cpu().numpy()
def log_intensity(field):
    arr = to_numpy_2d(intensity(field)); return np.log10(arr / (arr.max() + 1e-12) + 1e-8)
def wrapped_phase(field_or_phase):
    phase = torch.angle(field_or_phase) if torch.is_complex(field_or_phase) else field_or_phase
    return np.remainder(to_numpy_2d(phase), 2 * math.pi)
def overlay_experts(ax, color='lime'):
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        ax.add_patch(Rectangle((xs.start, ys.start), EXPERT_SIZE, EXPERT_SIZE, fill=False, edgecolor=color, linewidth=0.8))
        ax.text(xs.start + 5, ys.start + 18, EXPERT_IDS[idx], color=color, fontsize=7)
def overlay_prompt(ax, color='cyan'):
    if PROMPT_APERTURE_MODE == 'center_600': ax.add_patch(Rectangle((PADDING, PADDING), BANK_SIZE, BANK_SIZE, fill=False, edgecolor=color, linestyle='--', linewidth=1.0))
    else: ax.add_patch(Rectangle((0, 0), CANVAS_SIZE, CANVAS_SIZE, fill=False, edgecolor=color, linestyle='--', linewidth=1.0))
def show_intensity(field, title, filename=None, overlay='experts', figsize=(6, 6)):
    fig, ax = plt.subplots(figsize=figsize); im = ax.imshow(log_intensity(field), cmap='inferno')
    if overlay == 'experts': overlay_experts(ax)
    elif overlay == 'prompt': overlay_prompt(ax)
    ax.axhline(CANVAS_CENTER[0], color='white', alpha=0.18); ax.axvline(CANVAS_CENTER[1], color='white', alpha=0.18)
    ax.set_xlim(0, CANVAS_SIZE); ax.set_ylim(CANVAS_SIZE, 0); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label='log10(I/Imax+1e-8)')
    if SAVE_FIGURES and filename: fig.savefig(OUTPUT_DIR / filename, bbox_inches='tight')
    plt.show()
def show_phase(field_or_phase, title, filename=None, overlay='prompt', figsize=(6, 6)):
    fig, ax = plt.subplots(figsize=figsize); im = ax.imshow(wrapped_phase(field_or_phase), cmap='twilight', vmin=0, vmax=2*math.pi)
    if overlay == 'experts': overlay_experts(ax)
    elif overlay == 'prompt': overlay_prompt(ax)
    ax.set_xlim(0, CANVAS_SIZE); ax.set_ylim(CANVAS_SIZE, 0); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02, label='wrapped phase [0,2pi)')
    if SAVE_FIGURES and filename: fig.savefig(OUTPUT_DIR / filename, bbox_inches='tight')
    plt.show()
def show_heatmap_3x3(values, title, filename=None, cmap='magma'):
    arr = np.asarray(values, dtype=float).reshape(3, 3)
    fig, ax = plt.subplots(figsize=(4.8, 4.2)); im = ax.imshow(arr, cmap=cmap)
    for r in range(3):
        for c in range(3): ax.text(c, r, f'{arr[r,c]:.3f}', ha='center', va='center', color='white')
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2]); ax.set_title(title); fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    if SAVE_FIGURES and filename: fig.savefig(OUTPUT_DIR / filename, bbox_inches='tight')
    plt.show()

def expert_energy(field, active_threshold=0.02):
    I = intensity(field)[0]; vals = []
    for ys, xs in EXPERT_APERTURES: vals.append(float(I[ys, xs].sum().detach().cpu()))
    vals = np.asarray(vals, dtype=np.float64); expert_sum = vals.sum() + 1e-12
    total = float(I.sum().detach().cpu()) + 1e-12; outside = max(total - float(vals.sum()), 0.0)
    ratios = vals / expert_sum
    return {'ratios': ratios, 'raw': vals, 'outside_all_experts_energy_ratio': outside / total, 'active_expert_count': int((ratios > active_threshold).sum())}
def global_centroid(field):
    I = intensity(field)[0]; total = I.sum() + 1e-12
    yy = torch.arange(CANVAS_SIZE, device=device).float(); xx = torch.arange(CANVAS_SIZE, device=device).float(); YY, XX = torch.meshgrid(yy, xx, indexing='ij')
    return float((I * YY).sum().detach().cpu() / total.detach().cpu()), float((I * XX).sum().detach().cpu() / total.detach().cpu())
def per_expert_centroid(field):
    I = intensity(field)[0]; info = expert_energy(field); rows = []
    for idx, (ys, xs) in enumerate(EXPERT_APERTURES):
        crop = I[ys, xs]; e = crop.sum() + 1e-12
        yy = torch.arange(ys.start, ys.stop, device=device).float(); xx = torch.arange(xs.start, xs.stop, device=device).float(); YY, XX = torch.meshgrid(yy, xx, indexing='ij')
        cy = float((crop * YY).sum().detach().cpu() / e.detach().cpu()); cx = float((crop * XX).sum().detach().cpu() / e.detach().cpu())
        ty, tx = EXPERT_CENTERS[idx]; err = math.sqrt((cy - ty) ** 2 + (cx - tx) ** 2)
        rows.append({'expert_id': EXPERT_IDS[idx], 'energy_ratio': float(info['ratios'][idx]), 'centroid_y': cy, 'centroid_x': cx, 'target_y': float(ty), 'target_x': float(tx), 'center_error_px': err})
    return rows
def print_centroid_rows(rows, title):
    print('\n' + title); print('expert  energy   cy       cx       target_y target_x error_px')
    for r in rows: print(f"{r['expert_id']:>4s}  {r['energy_ratio']:.4f}  {r['centroid_y']:7.1f}  {r['centroid_x']:7.1f}  {r['target_y']:8.1f} {r['target_x']:8.1f} {r['center_error_px']:8.1f}")
    active = [r['center_error_px'] for r in rows if r['energy_ratio'] > 0.01]
    if active: print(f'active-region mean error: {np.mean(active):.2f} px, max error: {np.max(active):.2f} px')


## 5. Prompt Router Construction

三个 mode：complex_order_router 是主模式；phase_only_angle_sum 是 phase-only 对照；region_amplitude_angle_sum 是旧空间振幅分区思路对照。


In [ ]:
def build_amplitude_cases():
    cases = {'uniform':[1,1,1,1,1,1,1,1,1], 'center_only':[0,0,0,0,1,0,0,0,0]}
    for idx, eid in enumerate(EXPERT_IDS):
        v = [0] * 9; v[idx] = 1; cases[f'onehot_{eid}'] = v
    cases['diagonal'] = [1,0,0,0,1,0,0,0,1]
    cases['top_row'] = [1,1,1,0,0,0,0,0,0]
    cases['left_col'] = [1,0,0,1,0,0,1,0,0]
    cases['sparse_mix'] = [1.0,0,0.6,0,0.8,0,0.2,0,0]
    rng = np.random.default_rng(7); cases['random_seeded'] = rng.uniform(0.05, 1.0, size=9).round(3).tolist()
    cases['task_like_mnist'] = [1.0,0.2,0.0,0.1,0.8,0.1,0.0,0.2,0.7]
    cases['task_like_fashion'] = [0.8,1.0,0.8,0.2,0.7,0.2,0.0,0.1,0.0]
    cases['task_like_emnist'] = [0.8,0.0,0.1,0.9,0.4,0.0,0.8,0.0,0.5]
    return cases
AMPLITUDE_CASES = build_amplitude_cases()

def lens_phase():
    k = 2 * math.pi / WAVELENGTH_M; f = cm_to_m(FOCAL_LENGTH_CM)
    return -k / (2 * f) * (X_M**2 + Y_M**2)
def channel_table(scale=GRATING_SCALE, sign_x=GRATING_SIGN_X, sign_y=GRATING_SIGN_Y):
    rows = []; z2 = cm_to_m(PROMPT_TO_EXPERT_CM)
    for eid, (cy, cx) in zip(EXPERT_IDS, EXPERT_CENTERS):
        dx_px = cx - CANVAS_CENTER[1]; dy_px = cy - CANVAS_CENTER[0]
        fx = dx_px * PIXEL_SIZE_M / (WAVELENGTH_M * z2); fy = dy_px * PIXEL_SIZE_M / (WAVELENGTH_M * z2)
        fx_eff = sign_x * scale * fx; fy_eff = sign_y * scale * fy
        rows.append({'expert_id':eid, 'dx_px':dx_px, 'dy_px':dy_px, 'fx_cycles_per_m':fx_eff, 'fy_cycles_per_m':fy_eff, 'grating_period_x_px': math.inf if abs(fx_eff)<1e-12 else 1/(abs(fx_eff)*PIXEL_SIZE_M), 'grating_period_y_px': math.inf if abs(fy_eff)<1e-12 else 1/(abs(fy_eff)*PIXEL_SIZE_M), 'predicted_shift_x_px': z2*WAVELENGTH_M*fx_eff/PIXEL_SIZE_M, 'predicted_shift_y_px': z2*WAVELENGTH_M*fy_eff/PIXEL_SIZE_M})
    return rows
def print_channel_table(rows):
    print('expert dx dy fx fy period_x_px period_y_px pred_shift_x pred_shift_y')
    for r in rows:
        px = 'inf' if math.isinf(r['grating_period_x_px']) else f"{r['grating_period_x_px']:.2f}"; py = 'inf' if math.isinf(r['grating_period_y_px']) else f"{r['grating_period_y_px']:.2f}"
        print(f"{r['expert_id']:>3s} {r['dx_px']:>4.0f} {r['dy_px']:>4.0f} {r['fx_cycles_per_m']:>10.1f} {r['fy_cycles_per_m']:>10.1f} {px:>8s} {py:>8s} {r['predicted_shift_x_px']:>8.1f} {r['predicted_shift_y_px']:>8.1f}")
print_channel_table(channel_table())
save_csv(OUTPUT_DIR / 'theoretical_channel_table.csv', channel_table())

def grating_phase_bank(scale=GRATING_SCALE, sign_x=GRATING_SIGN_X, sign_y=GRATING_SIGN_Y):
    return [2 * math.pi * (r['fx_cycles_per_m'] * X_M + r['fy_cycles_per_m'] * Y_M) for r in channel_table(scale, sign_x, sign_y)]
def region_amplitude_map(amplitudes):
    amap = torch.zeros(CANVAS_SHAPE, dtype=torch.float32, device=device)
    for amp, (ys, xs) in zip(amplitudes, PROMPT_APERTURES): amap[ys, xs] = float(amp)
    return amap
def normalize_complex_amplitude(router): return router / max(float(torch.abs(router).max().detach().cpu()), 1.0)
def build_prompt_transmission(amplitude_case='uniform', prompt_mode=PROMPT_MODE, scale=GRATING_SCALE, sign_x=GRATING_SIGN_X, sign_y=GRATING_SIGN_Y, aperture_mode=PROMPT_APERTURE_MODE):
    amplitudes = torch.tensor(AMPLITUDE_CASES[amplitude_case], dtype=torch.float32, device=device)
    phases = grating_phase_bank(scale, sign_x, sign_y); aperture = prompt_aperture_mask(aperture_mode).float()
    lens = torch.exp(1j * lens_phase()).to(torch.complex64)
    if prompt_mode == 'complex_order_router':
        router = torch.zeros(CANVAS_SHAPE, dtype=torch.complex64, device=device)
        for amp, phase in zip(amplitudes, phases): router = router + amp.to(torch.complex64) * torch.exp(1j * phase).to(torch.complex64)
        router = normalize_complex_amplitude(router); total = aperture.to(torch.complex64) * lens * router
    elif prompt_mode == 'phase_only_angle_sum':
        router_sum = torch.zeros(CANVAS_SHAPE, dtype=torch.complex64, device=device)
        for amp, phase in zip(amplitudes, phases): router_sum = router_sum + amp.to(torch.complex64) * torch.exp(1j * phase).to(torch.complex64)
        router = torch.exp(1j * torch.angle(router_sum)).to(torch.complex64); total = aperture.to(torch.complex64) * lens * router
    elif prompt_mode == 'region_amplitude_angle_sum':
        router_sum = torch.zeros(CANVAS_SHAPE, dtype=torch.complex64, device=device)
        for phase in phases: router_sum = router_sum + torch.exp(1j * phase).to(torch.complex64)
        router = torch.exp(1j * torch.angle(router_sum)).to(torch.complex64); total = aperture.to(torch.complex64) * region_amplitude_map(amplitudes.detach().cpu().tolist()).to(torch.complex64) * lens * router
    else: raise ValueError(prompt_mode)
    return {'prompt_transmission': total.to(torch.complex64), 'lens_phase': lens_phase() * aperture, 'router': router, 'router_amplitude': torch.abs(router) * aperture, 'router_phase': torch.angle(router) * aperture, 'total_amplitude': torch.abs(total), 'total_phase': torch.angle(total) * aperture, 'amplitudes': amplitudes.detach().cpu().numpy(), 'scale': scale, 'sign_x': sign_x, 'sign_y': sign_y}


## 6. Onehot Grating Calibration

对 scale/sign 做 onehot centroid sweep。每个候选都运行完整角谱传播 input -> prompt -> expert。


In [ ]:
def run_as_router(input_field, amplitude_case='uniform', prompt_mode=PROMPT_MODE, scale=GRATING_SCALE, sign_x=GRATING_SIGN_X, sign_y=GRATING_SIGN_Y, return_layers=False):
    prompt = build_prompt_transmission(amplitude_case, prompt_mode, scale, sign_x, sign_y)
    field_before_prompt = prop_input_to_prompt(input_field)
    field_after_prompt = field_before_prompt * prompt['prompt_transmission'].unsqueeze(0)
    expert_entrance = prop_prompt_to_expert(field_after_prompt)
    result = {'field_before_prompt': field_before_prompt, 'field_after_prompt': field_after_prompt, 'expert_entrance': expert_entrance, 'prompt': prompt, 'expert_energy': expert_energy(expert_entrance), 'per_expert_centroid': per_expert_centroid(expert_entrance), 'global_centroid': global_centroid(expert_entrance)}
    if return_layers:
        layers = []; field = expert_entrance * EXPERT_UNION_MASK.unsqueeze(0).to(torch.complex64); layers.append(field)
        for _ in range(NUM_IDENTITY_EXPERT_LAYERS):
            field = prop_inter_layer(field); field = field * EXPERT_UNION_MASK.unsqueeze(0).to(torch.complex64); layers.append(field)
        result['layers'] = layers; result['detector'] = layers[-1]; result['detector_energy'] = expert_energy(layers[-1])
    return result

def target_error_for_case(field, target_idx):
    cy, cx = global_centroid(field); ty, tx = EXPERT_CENTERS[target_idx]
    return math.sqrt((cy - ty)**2 + (cx - tx)**2), cy, cx

def calibrate_grating():
    candidate_scales = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
    candidate_signs = [(-1,-1), (-1,+1), (+1,-1), (+1,+1)]
    calibration_input = embed_input(make_input_image('flat_top_200', INPUT_SIZE).to(device))
    rows = []; best = None
    for scale in candidate_scales:
        for sign_x, sign_y in candidate_signs:
            errors = []; top_hits = 0
            for idx, eid in enumerate(EXPERT_IDS):
                res = run_as_router(calibration_input, f'onehot_{eid}', 'complex_order_router', scale, sign_x, sign_y, False)
                err, cy, cx = target_error_for_case(res['expert_entrance'], idx)
                ratios = res['expert_energy']['ratios']; top_idx = int(np.argmax(ratios)); top_hits += int(top_idx == idx); errors.append(err)
                rows.append({'scale': scale, 'sign_x': sign_x, 'sign_y': sign_y, 'target_expert': eid, 'centroid_y': cy, 'centroid_x': cx, 'target_y': EXPERT_CENTERS[idx][0], 'target_x': EXPERT_CENTERS[idx][1], 'centroid_error_px': err, 'target_energy_ratio': float(ratios[idx]), 'top_expert': EXPERT_IDS[top_idx], 'top_is_target': bool(top_idx == idx)})
            summary = {'scale': scale, 'sign_x': sign_x, 'sign_y': sign_y, 'mean_centroid_error_px': float(np.mean(errors)), 'max_centroid_error_px': float(np.max(errors)), 'diagonal_top_count': int(top_hits)}
            if best is None or summary['mean_centroid_error_px'] < best['mean_centroid_error_px']: best = summary
    save_csv(OUTPUT_DIR / 'grating_calibration_table.csv', rows); save_json(OUTPUT_DIR / 'best_grating_calibration.json', best)
    print('Best grating calibration:'); print(json.dumps(best, indent=2)); return best, rows

if RUN_CALIBRATION:
    BEST_CALIBRATION, CALIBRATION_ROWS = calibrate_grating()
else:
    BEST_CALIBRATION = {'scale': GRATING_SCALE, 'sign_x': GRATING_SIGN_X, 'sign_y': GRATING_SIGN_Y, 'mean_centroid_error_px': None, 'max_centroid_error_px': None, 'diagonal_top_count': None}; CALIBRATION_ROWS = []
    print('Calibration skipped; using default scale/sign:', BEST_CALIBRATION)
GRATING_SCALE_USED = float(BEST_CALIBRATION['scale']); GRATING_SIGN_X_USED = float(BEST_CALIBRATION['sign_x']); GRATING_SIGN_Y_USED = float(BEST_CALIBRATION['sign_y'])
print_channel_table(channel_table(GRATING_SCALE_USED, GRATING_SIGN_X_USED, GRATING_SIGN_Y_USED))


In [ ]:
def plot_calibration_overlay(rows=CALIBRATION_ROWS):
    if not rows: print('No calibration rows to plot.'); return
    best_rows = [r for r in rows if float(r['scale']) == GRATING_SCALE_USED and float(r['sign_x']) == GRATING_SIGN_X_USED and float(r['sign_y']) == GRATING_SIGN_Y_USED]
    fig, ax = plt.subplots(figsize=(7, 7)); ax.set_facecolor('black'); overlay_experts(ax)
    for r in best_rows:
        ax.plot(r['centroid_x'], r['centroid_y'], 'ro', markersize=5); ax.plot(r['target_x'], r['target_y'], 'cx', markersize=7)
        ax.plot([r['centroid_x'], r['target_x']], [r['centroid_y'], r['target_y']], color='yellow', alpha=0.5, linewidth=0.8)
    ax.set_xlim(0, CANVAS_SIZE); ax.set_ylim(CANVAS_SIZE, 0); ax.set_title('Calibration centroid overlay: red=measured, cyan=target')
    if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'calibration_centroid_overlay.png', bbox_inches='tight')
    plt.show()
plot_calibration_overlay()


## 7. Main Run: Uniform Routing


In [ ]:
main_result = run_as_router(input_field, AMPLITUDE_CASE, PROMPT_MODE, GRATING_SCALE_USED, GRATING_SIGN_X_USED, GRATING_SIGN_Y_USED, return_layers=True)
show_intensity(input_field, 'input plane intensity', 'input_plane_intensity.png', overlay='experts')
show_intensity(main_result['field_before_prompt'], 'before prompt after angular-spectrum propagation z1', 'before_prompt_intensity.png', overlay='prompt')
show_phase(main_result['prompt']['lens_phase'], 'prompt lens phase wrapped', 'prompt_lens_phase_wrapped.png', overlay='prompt')
show_intensity(main_result['prompt']['router_amplitude'].unsqueeze(0), 'prompt router amplitude', 'prompt_router_amplitude.png', overlay='prompt')
show_phase(main_result['prompt']['router_phase'], 'prompt router phase wrapped', 'prompt_router_phase_wrapped.png', overlay='prompt')
show_intensity(main_result['prompt']['total_amplitude'].unsqueeze(0), 'prompt total amplitude', 'prompt_total_amplitude.png', overlay='prompt')
show_phase(main_result['prompt']['total_phase'], 'prompt total phase wrapped', 'prompt_total_phase_wrapped.png', overlay='prompt')
show_intensity(main_result['field_after_prompt'], 'after prompt intensity', 'after_prompt_intensity.png', overlay='prompt')
show_intensity(main_result['expert_entrance'], 'expert entrance from AngularSpectrumPropagator', 'expert_entrance_intensity.png', overlay='experts')
show_heatmap_3x3(main_result['expert_energy']['ratios'], 'expert entrance energy ratio', 'expert_entrance_energy_3x3.png')
show_intensity(main_result['layers'][0], 'after expert aperture intensity', 'after_expert_aperture_intensity.png', overlay='experts')
show_intensity(main_result['layers'][1], 'identity layer 1 intensity', 'identity_layer_1_intensity.png', overlay='experts')
show_intensity(main_result['layers'][min(3, len(main_result['layers'])-1)], 'identity layer 3 intensity', 'identity_layer_3_intensity.png', overlay='experts')
show_intensity(main_result['detector'], 'final detector intensity after identity propagation', 'final_detector_intensity.png', overlay='experts')
show_heatmap_3x3(main_result['detector_energy']['ratios'], 'detector energy ratio', 'detector_energy_3x3.png')
print_centroid_rows(main_result['per_expert_centroid'], 'expert entrance per-expert centroid')


## 8. Amplitude Control Tests

这些测试检查 a_j 在 prompt 平面进入 R_router 后，expert energy 是否和 commanded power 相关。


In [ ]:
def vector_metrics(commanded, measured):
    commanded = np.asarray(commanded, dtype=np.float64); measured = np.asarray(measured, dtype=np.float64)
    cosine = float(np.dot(commanded, measured) / ((np.linalg.norm(commanded) * np.linalg.norm(measured)) + 1e-12))
    rmse = float(np.sqrt(np.mean((commanded - measured) ** 2)))
    pearson = float('nan') if np.std(commanded) < 1e-12 or np.std(measured) < 1e-12 else float(np.corrcoef(commanded, measured)[0, 1])
    return cosine, rmse, pearson
def commanded_power(amplitudes):
    arr = np.asarray(amplitudes, dtype=np.float64) ** 2; return arr / (arr.sum() + 1e-12)
def run_amplitude_control_tests():
    cases = ['uniform', 'center_only', 'diagonal', 'top_row', 'left_col', 'sparse_mix', 'task_like_mnist', 'task_like_fashion', 'task_like_emnist']
    rows = []
    for case in cases:
        res = run_as_router(input_field, case, PROMPT_MODE, GRATING_SCALE_USED, GRATING_SIGN_X_USED, GRATING_SIGN_Y_USED)
        measured = res['expert_energy']['ratios']; cmd = commanded_power(AMPLITUDE_CASES[case]); cosine, rmse, pearson = vector_metrics(cmd, measured)
        row = {'amplitude_case': case, 'cosine_commanded_measured': cosine, 'rmse_commanded_measured': rmse, 'pearson_commanded_measured': pearson, 'outside_all_experts_energy_ratio': res['expert_energy']['outside_all_experts_energy_ratio']}
        for eid, c, m in zip(EXPERT_IDS, cmd, measured): row[f'commanded_power_{eid}'] = float(c); row[f'measured_energy_{eid}'] = float(m)
        rows.append(row); print(case, 'cosine=', round(cosine, 3), 'rmse=', round(rmse, 3), 'pearson=', pearson)
    save_csv(OUTPUT_DIR / 'amplitude_control_table.csv', rows); return rows
if RUN_AMPLITUDE_TESTS: AMPLITUDE_CONTROL_ROWS = run_amplitude_control_tests()
else: AMPLITUDE_CONTROL_ROWS = []; print('Amplitude control tests skipped.')


In [ ]:
def plot_amplitude_vs_measured(rows=AMPLITUDE_CONTROL_ROWS):
    if not rows: return
    selected = rows[:min(6, len(rows))]; fig, axes = plt.subplots(len(selected), 1, figsize=(9, 2.2 * len(selected)))
    if len(selected) == 1: axes = [axes]
    x = np.arange(9)
    for ax, row in zip(axes, selected):
        cmd = [row[f'commanded_power_{e}'] for e in EXPERT_IDS]; meas = [row[f'measured_energy_{e}'] for e in EXPERT_IDS]
        ax.bar(x - 0.18, cmd, width=0.36, label='commanded power'); ax.bar(x + 0.18, meas, width=0.36, label='measured energy')
        ax.set_xticks(x); ax.set_xticklabels(EXPERT_IDS); ax.set_ylim(0, max(max(cmd), max(meas), 0.01) * 1.25)
        ax.set_title(f"{row['amplitude_case']} | cosine={row['cosine_commanded_measured']:.3f}"); ax.legend(loc='upper right')
    fig.tight_layout()
    if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'amplitude_vs_measured_bar.png', bbox_inches='tight')
    plt.show()
plot_amplitude_vs_measured()


## 9. Onehot Routing Matrix


In [ ]:
def run_onehot_routing_matrix():
    matrix = []; rows = []
    for idx, eid in enumerate(EXPERT_IDS):
        res = run_as_router(input_field, f'onehot_{eid}', PROMPT_MODE, GRATING_SCALE_USED, GRATING_SIGN_X_USED, GRATING_SIGN_Y_USED)
        measured = res['expert_energy']['ratios']; matrix.append(measured)
        row = {'commanded_expert': eid}
        for out_id, val in zip(EXPERT_IDS, measured): row[out_id] = float(val)
        row['top_expert'] = EXPERT_IDS[int(np.argmax(measured))]; row['top_is_commanded'] = bool(int(np.argmax(measured)) == idx); rows.append(row)
    save_csv(OUTPUT_DIR / 'onehot_routing_matrix.csv', rows); return np.asarray(matrix), rows
if RUN_ONEHOT_MATRIX:
    ONEHOT_MATRIX, ONEHOT_ROWS = run_onehot_routing_matrix()
    fig, ax = plt.subplots(figsize=(6, 5.5)); im = ax.imshow(ONEHOT_MATRIX, cmap='magma', vmin=0, vmax=max(ONEHOT_MATRIX.max(), 1e-6))
    ax.set_xticks(range(9)); ax.set_xticklabels(EXPERT_IDS, rotation=45); ax.set_yticks(range(9)); ax.set_yticklabels(EXPERT_IDS)
    ax.set_xlabel('measured expert energy'); ax.set_ylabel('commanded prompt route'); ax.set_title('onehot routing matrix'); fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'onehot_routing_matrix.png', bbox_inches='tight')
    plt.show()
else: ONEHOT_MATRIX, ONEHOT_ROWS = None, []; print('Onehot routing matrix skipped.')


## 10. Same-Input Copy Similarity

对 uniform case，把 9 个 expert patch 的 intensity 裁出来，比较它们是否像同一输入的 9 个副本。


In [ ]:
def normalized_patch_vectors(field):
    I = intensity(field)[0]; vecs = []
    for ys, xs in EXPERT_APERTURES:
        crop = I[ys, xs].detach().cpu().numpy().astype(np.float64); crop = crop / (crop.sum() + 1e-12); vecs.append(crop.reshape(-1))
    return vecs
def copy_similarity(field):
    vecs = normalized_patch_vectors(field); rows = []; cosine_mat = np.zeros((9,9)); corr_mat = np.zeros((9,9)); nrmse_mat = np.zeros((9,9))
    for i in range(9):
        for j in range(9):
            a, b = vecs[i], vecs[j]
            cosine = float(np.dot(a,b) / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-12))
            corr = float('nan') if np.std(a)<1e-12 or np.std(b)<1e-12 else float(np.corrcoef(a,b)[0,1])
            nrmse = float(np.sqrt(np.mean((a-b)**2)) / (np.sqrt(np.mean(a**2))+1e-12))
            cosine_mat[i,j] = cosine; corr_mat[i,j] = corr; nrmse_mat[i,j] = nrmse; rows.append({'expert_a':EXPERT_IDS[i], 'expert_b':EXPERT_IDS[j], 'cosine':cosine, 'correlation':corr, 'nrmse':nrmse})
    save_csv(OUTPUT_DIR / 'expert_patch_similarity.csv', rows); return cosine_mat, corr_mat, nrmse_mat, rows
if RUN_COPY_SIMILARITY:
    COSINE_MAT, CORR_MAT, NRMSE_MAT, SIM_ROWS = copy_similarity(main_result['expert_entrance'])
    fig, ax = plt.subplots(figsize=(6, 5.5)); im = ax.imshow(CORR_MAT, cmap='viridis', vmin=0, vmax=1)
    ax.set_xticks(range(9)); ax.set_xticklabels(EXPERT_IDS, rotation=45); ax.set_yticks(range(9)); ax.set_yticklabels(EXPERT_IDS); ax.set_title('expert patch similarity: Pearson correlation')
    fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'expert_patch_similarity_heatmap.png', bbox_inches='tight')
    plt.show()
else: COSINE_MAT, CORR_MAT, NRMSE_MAT, SIM_ROWS = None, None, None, []; print('Copy similarity skipped.')


## 11. Identity Expert Propagation Drift

如果 expert entrance 已对齐，但后续 identity propagation 中边缘或角落专家快速漂移，说明到达 expert 后仍有残余倾角。此处只记录，不在 expert 面直接补偿。


In [ ]:
def phase_gradient_by_expert(field):
    rows = []; U = field[0]
    for eid, (ys, xs) in zip(EXPERT_IDS, EXPERT_APERTURES):
        crop = U[ys, xs]
        Ix = torch.abs(crop[:,1:]).square(); Iy = torch.abs(crop[1:,:]).square()
        gx = torch.angle(crop[:,1:] * torch.conj(crop[:,:-1])); gy = torch.angle(crop[1:,:] * torch.conj(crop[:-1,:]))
        rows.append({'expert_id': eid, 'mean_phase_gradient_x_rad_per_px': float((gx*Ix).sum().detach().cpu()/(Ix.sum().detach().cpu()+1e-12)), 'mean_phase_gradient_y_rad_per_px': float((gy*Iy).sum().detach().cpu()/(Iy.sum().detach().cpu()+1e-12))})
    return rows
def analyze_drift(layers):
    rows = []; entrance = {r['expert_id']: r for r in per_expert_centroid(layers[0])}
    for layer_idx, field in enumerate(layers):
        energy = expert_energy(field)
        for r in per_expert_centroid(field):
            base = entrance[r['expert_id']]; drift = math.sqrt((r['centroid_y']-base['centroid_y'])**2 + (r['centroid_x']-base['centroid_x'])**2)
            rows.append({'layer_idx': layer_idx, 'expert_id': r['expert_id'], 'energy_ratio': r['energy_ratio'], 'centroid_y': r['centroid_y'], 'centroid_x': r['centroid_x'], 'drift_from_entrance_px': drift, 'outside_all_experts_energy_ratio': energy['outside_all_experts_energy_ratio']})
    save_csv(OUTPUT_DIR / 'centroid_drift_by_layer.csv', rows); return rows
if RUN_DRIFT_ANALYSIS:
    DRIFT_ROWS = analyze_drift(main_result['layers']); PHASE_GRADIENT_ROWS = phase_gradient_by_expert(main_result['expert_entrance']); save_csv(OUTPUT_DIR / 'local_phase_gradient_by_expert.csv', PHASE_GRADIENT_ROWS)
    by_layer = {}
    for r in DRIFT_ROWS:
        if r['energy_ratio'] > 0.01: by_layer.setdefault(r['layer_idx'], []).append(r['drift_from_entrance_px'])
    layer_ids = sorted(by_layer); max_drift = [max(by_layer[i]) if by_layer[i] else 0 for i in layer_ids]; mean_drift = [float(np.mean(by_layer[i])) if by_layer[i] else 0 for i in layer_ids]
    fig, ax = plt.subplots(figsize=(6,4)); ax.plot(layer_ids, mean_drift, marker='o', label='mean active drift'); ax.plot(layer_ids, max_drift, marker='s', label='max active drift')
    ax.set_xlabel('identity propagation layer index'); ax.set_ylabel('drift from entrance [px]'); ax.set_title('centroid drift through identity expert propagation'); ax.legend(); ax.grid(True, alpha=0.25)
    if SAVE_FIGURES: fig.savefig(OUTPUT_DIR / 'centroid_drift_summary.png', bbox_inches='tight')
    plt.show()
else: DRIFT_ROWS = []; PHASE_GRADIENT_ROWS = []; print('Drift analysis skipped.')


## 12. Summary and Interpretation


In [ ]:
def status_from_threshold(value, pass_thr, warn_thr, lower_is_better=True):
    if value is None or (isinstance(value, float) and math.isnan(value)): return 'N/A'
    if lower_is_better: return 'PASS' if value < pass_thr else ('WARN' if value < warn_thr else 'FAIL')
    return 'PASS' if value >= pass_thr else ('WARN' if value >= warn_thr else 'FAIL')
uniform_active_count = int(main_result['expert_energy']['active_expert_count'])
uniform_active_status = 'PASS' if uniform_active_count >= 7 else ('WARN' if uniform_active_count >= 5 else 'FAIL')
onehot_diag_count = int(sum(bool(r['top_is_commanded']) for r in ONEHOT_ROWS)) if ONEHOT_ROWS else None
onehot_status = 'N/A' if onehot_diag_count is None else ('PASS' if onehot_diag_count >= 7 else ('WARN' if onehot_diag_count >= 5 else 'FAIL'))
mean_centroid_error = BEST_CALIBRATION.get('mean_centroid_error_px')
centroid_status = status_from_threshold(mean_centroid_error, 30, 60, True)
mean_pairwise_corr = None; copy_similarity_status = 'N/A'
if CORR_MAT is not None:
    mean_pairwise_corr = float(np.nanmean(CORR_MAT[~np.eye(9, dtype=bool)])); copy_similarity_status = status_from_threshold(mean_pairwise_corr, 0.80, 0.60, False)
max_drift = None; drift_status = 'N/A'
if DRIFT_ROWS:
    active_drifts = [r['drift_from_entrance_px'] for r in DRIFT_ROWS if r['layer_idx'] > 0 and r['energy_ratio'] > 0.01]
    if active_drifts: max_drift = float(max(active_drifts)); drift_status = status_from_threshold(max_drift, 35, 70, True)
conclusions = []
conclusions.append('AS global router routes onehot channels to expected apertures.' if onehot_status == 'PASS' and centroid_status == 'PASS' else 'Inspect grating sign/scale if onehot routing or centroid alignment is not PASS.')
conclusions.append('Uniform prompt activates most expert apertures.' if uniform_active_status == 'PASS' else 'Uniform prompt does not illuminate enough experts.')
if copy_similarity_status != 'N/A': conclusions.append('Expert patches look like similar routed copies.' if copy_similarity_status == 'PASS' else 'Copy similarity is limited; inspect side lobes or residual tilt.')
if drift_status != 'N/A': conclusions.append('Identity propagation is stable.' if drift_status == 'PASS' else 'Later propagation drifts; compensation should be encoded back into prompt plane.')
summary = {'notebook':'nine_expert_as_global_router_step_by_step.ipynb', 'uses_angular_spectrum_for_expert_entrance': True, 'prompt_mode': PROMPT_MODE, 'input_type': INPUT_TYPE, 'amplitude_case': AMPLITUDE_CASE, 'geometry': {'wavelength_m': WAVELENGTH_M, 'pixel_size_m': PIXEL_SIZE_M, 'input_size': INPUT_SIZE, 'expert_size': EXPERT_SIZE, 'expert_pitch': EXPERT_PITCH, 'canvas_shape': CANVAS_SHAPE, 'input_to_prompt_cm': INPUT_TO_PROMPT_CM, 'prompt_to_expert_cm': PROMPT_TO_EXPERT_CM, 'focal_length_cm': FOCAL_LENGTH_CM, 'magnification': magnification, 'expected_image_size_px': expected_image_size_px, 'theoretical_grating_period_for_pitch_px': period_pitch_px}, 'best_calibration': BEST_CALIBRATION, 'checks': {'onehot_routing': {'status': onehot_status, 'diagonal_top_count': onehot_diag_count}, 'centroid_alignment': {'status': centroid_status, 'mean_error_px': mean_centroid_error}, 'uniform_active_experts': {'status': uniform_active_status, 'active_expert_count': uniform_active_count}, 'copy_similarity': {'status': copy_similarity_status, 'mean_pairwise_correlation': mean_pairwise_corr}, 'propagation_drift': {'status': drift_status, 'max_drift_px': max_drift}}, 'conclusions': conclusions}
save_json(OUTPUT_DIR / 'summary.json', summary)
print(json.dumps(summary, indent=2))
for item in conclusions: print('-', item)
